# Retrieval Methods

## Load Secrets

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## Index both Dense and Sparse Embeddings

In [5]:
import os
import cohere
import pymupdf4llm
from fastembed import SparseTextEmbedding
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    Modifier,
    PointStruct,
    SparseVector,
    SparseVectorParams,
    VectorParams,
)

co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])
qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
)
COLLECTION = "week2_article"
PDF_FILE_PATH = "./data/article.pdf"

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_text(pymupdf4llm.to_markdown(PDF_FILE_PATH))

dense = co.embed(
    model="embed-v4.0",
    input_type="search_document",
    embedding_types=["float"],
    texts=chunks,
).embeddings.float

bm25 = SparseTextEmbedding("Qdrant/bm25")
sparse = [
    SparseVector(indices=sv.indices.tolist(), values=sv.values.tolist())
    for sv in bm25.embed(chunks)
]

qdrant.recreate_collection(
    collection_name=COLLECTION,
    vectors_config={
        "dense": VectorParams(size=len(dense[0]), distance=Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams(modifier=Modifier.IDF),
    },
)

qdrant.upsert(
    collection_name=COLLECTION,
    points=[
        PointStruct(
            id=i,
            vector={"dense": dense[i], "sparse": sparse[i]},
            payload={"text": chunks[i], "source": PDF_FILE_PATH},
        )
        for i in range(len(chunks))
    ],
)

print(
    f"Reindexed — {qdrant.get_collection(COLLECTION).points_count} points with dense+sparse."
)

C:\Users\Ashutosh Bhadoria\AppData\Local\Temp\ipykernel_21232\1481954913.py:44: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


Reindexed — 91 points with dense+sparse.


## Utility Functions

### Dense Seach

In [17]:
def search_dense(query: str, k: int=5):
    q = co.embed(
        model="embed-v4.0",
        input_type="search_query",
        embedding_types=["float"],
        texts=[query],
    ).embeddings.float[0]

    return qdrant.query_points(
        collection_name=COLLECTION,
        query=q,
        using="dense",
        limit=k,
    ).points



### Sparse Search

In [ ]:
def search_sparse(query: str, k: int = 5):
    sv = next(iter(bm25.query_embed([query])))
    q = SparseVector(indices=sv.indices.tolist(), values=sv.values.tolist())
    return qdrant.query_points(
        collection_name=COLLECTION,
        query=q,
        using="sparse",
        limit=k,
    ).points

### Hybrid Search

In [14]:
from qdrant_client.models import FusionQuery, Prefetch


def search_hybrid(query: str, k: int = 5, fetch: int = 20):
    q_dense = co.embed(
        model="embed-v4.0",
        input_type="search_query",
        embedding_types=["float"],
        texts=[query],
    ).embeddings.float[0]

    sv = next(iter(bm25.embed([query])))
    q_sparse = SparseVector(indices=sv.indices.tolist(), values=sv.values.tolist())

    return qdrant.query_points(
        collection_name=COLLECTION,
        prefetch=[
            Prefetch(
                query=q_dense,
                using="dense",
                limit=fetch,
            ),
            Prefetch(
                query=q_sparse,
                using="sparse",
                limit=fetch,
            ),
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k,
    ).points

### Display Search Results

In [15]:
def show(label, hits):
    print(f"\n--- {label} ---")
    for h in hits[:3]:
        snippet = h.payload["text"][:90].replace("\n", " ")
        print(f"  {h.score:.3f}  {snippet}…")

## Test

### Semantic-heavy search

In [18]:
q1 = "Tell me some of the common problems of the model layer"
show("DENSE  " + repr(q1), search_dense(q1))
show("SPARSE " + repr(q1), search_sparse(q1))
show("HYBRID " + repr(q1), search_hybrid(q1))


--- DENSE  'Tell me some of the common problems of the model layer' ---
  0.405  ## **§2 - The Model Layer**   **==> picture [414 x 169] intentionally omitted <==**  Let m…
  0.358  This is the lesson I wish someone had screamed at me on day one:   ## **The agent doesn't …
  0.341  You know the one:   **==> picture [414 x 148] intentionally omitted <==**  The point was b…

--- SPARSE 'Tell me some of the common problems of the model layer' ---
  11.213  Vibes don't compose. Vibes don't survive a model update. Vibes can't tell you whether your…
  7.210  ## **Aggressive retry on transient failures, with corrective feedback**   ("your last tool…
  7.175  Fine, knock yourself out. But the bug that wakes you up at 3am is almost always in the **s…

--- HYBRID 'Tell me some of the common problems of the model layer' ---
  0.625  ## **§2 - The Model Layer**   **==> picture [414 x 169] intentionally omitted <==**  Let m…
  0.500  Vibes don't compose. Vibes don't survive a model update. Vibes c

### Keyword-heavy search

In [19]:
RARE_TOKEN = "LangGraph"

show("DENSE  " + repr(RARE_TOKEN), search_dense(RARE_TOKEN))
show("SPARSE " + repr(RARE_TOKEN), search_sparse(RARE_TOKEN))
show("HYBRID " + repr(RARE_TOKEN), search_hybrid(RARE_TOKEN))


--- DENSE  'LangGraph' ---
  0.406  And don't only log the final output — when the answer is wrong, the bug is in steps 3, 7, …
  0.324  Sometimes it's explicit — a LangGraph DAG with named nodes, edges, and conditionals you ca…
  0.267  ## **§3 - Agent Harness**   **==> picture [414 x 171] intentionally omitted <==**  Vivek T…

--- SPARSE 'LangGraph' ---
  5.493  Sometimes it's explicit — a LangGraph DAG with named nodes, edges, and conditionals you ca…
  5.401  And don't only log the final output — when the answer is wrong, the bug is in steps 3, 7, …

--- HYBRID 'LangGraph' ---
  0.833  Sometimes it's explicit — a LangGraph DAG with named nodes, edges, and conditionals you ca…
  0.833  And don't only log the final output — when the answer is wrong, the bug is in steps 3, 7, …
  0.250  ## **§3 - Agent Harness**   **==> picture [414 x 171] intentionally omitted <==**  Vivek T…


## Recap

| | Strengths | Weaknesses |
|---|---|---|
| **Dense** | Synonyms, paraphrase, meaning | Rare tokens, exact symbols |
| **Sparse (BM25)** | Rare tokens, code/IDs | Blind to synonyms |
| **Hybrid (RRF)** | Both | Slightly more candidates to score |

Hybrid is what the deployed agent uses.